In [1]:

import glob #sirve para encontrar archivos dentro de una ruta, con una extension especificada

# Leer DATASET de Train

train_raw = []
print("Leyendo set de entrenamiento...\n")
for filename in glob.glob("./datasets-2026/datasets/train/*.tsv"):
    print(filename)
    print('\t'+' _ _'*25)
    with open(filename,'r', encoding='utf-8') as f:
        for line in f:
            train_raw.append(line)
        
print(f"\tTotal de datos almacenados {len(train_raw)}")
        

Leyendo set de entrenamiento...

./datasets-2026/datasets/train\cr.tsv
	 _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
./datasets-2026/datasets/train\es.tsv
	 _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
./datasets-2026/datasets/train\mx.tsv
	 _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
./datasets-2026/datasets/train\pe.tsv
	 _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
./datasets-2026/datasets/train\uy.tsv
	 _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
	Total de datos almacenados 4802


In [ ]:
# Leer DATASET de Prueba

test_raw = []
print("Leyendo set de prueba...\n")
for filename in glob.glob("./datasets-2026/datasets/dev/*.tsv"):
    print(filename)
    print('\t'+' _ _'*25)
    with open(filename,'r', encoding='utf-8') as f:
        for line in f:
            test_raw.append(line)
        
print(f"\tTotal de datos almacenados {len(test_raw)}")

In [ ]:
print(train_raw[0])
print(train_raw[200])
print(test_raw[2000])

# 4.1 Pre-procesamiento con regex

In [ ]:
import re

def extraer_categoria(tweet):
    if re.search('N$', tweet):
        cat = 'negativo'
    if re.search('NEU$', tweet):
        cat = 'neutral'
    if re.search('P$', tweet):
        cat = 'positivo'
    return cat

    
idx = 2004
print(f'Tweet: {train_raw[idx]}')
print(f'Categoria: {extraer_categoria(train_raw[idx])}')



# 4.2 Eliminar identificador y caracteres no importante
 #### Patrones de textos a eliminar

In [ ]:
# \d{18}\t    -----> cualquier numerico de 18 caracteres mas tabulacion.
# @\w+        -----> cualquier alfanumerico precedido por una @.
# #\w+        -----> cualquier alfanumerico precedido por una #.
# \w*\d\w*    -----> cualquier alfanumerico con numeros dentro (letra y numeros mezclados).
# [\\¿?¡!$'"&%()*{};@:.,'\/]+     -----> cualquier otro signo especial.
# ['¿?¡!$#\\\/]+

In [ ]:
# los backslashes doble (\\) son usado para que python no lo interprete el \d, \w, etc... como tabulacion, salto de lineas, etc...
def limpiar_tweet(tweet):
    tweet = re.sub("\\d{18}\\t", "", tweet)
    tweet = re.sub("@\\w+", "", tweet)
    tweet = re.sub("#\\w+", "", tweet)
    #tweet = re.sub("\\w*\\d\\w*", "", tweet)  #linea fue comentada porque me elimina los numeros de los tweets
    tweet = re.sub("\n", "", tweet)
    tweet = re.sub("\tN$","", tweet)
    tweet = re.sub("\tNEU$","", tweet)
    tweet = re.sub("\tP$","", tweet)
    tweet = re.sub("[¿?¡!$'#]+", "", tweet) #faltan tratar estos signos: /\|
    tweet.strip()
    tweet.lower()    

    return tweet
    
#print(train_raw[200])
#print(limpiar_tweet(train_raw[200]))


In [ ]:
#Ahora si, pre-procesamos los dataset  (Se crea una lista con los Tweets limpios y otra lista solo con las categorias)

x_train = list(map(limpiar_tweet, train_raw)) # train tweets
x_test = list(map(limpiar_tweet, test_raw))   # test tweets

y_train = list(map(extraer_categoria, train_raw)) # train categorias
y_test = list(map(extraer_categoria, test_raw))   # test categorias


In [ ]:
# cantidad de veces que se repite cada palabras en train_raw (una lista)
def dic_stop_words(tweets): #los tweets deben estar en una lista, esto porque que el primer FOR itera una lista y el segundo sobre el String
    #crear el diccionario
    dicc = {}

    for tweet in tweets:
        for word in tweet.split(' '):
            if word in dicc:
                dicc[word]+=1
            else:
                dicc[word]=1

    # ORDENAR de forma descendiente (Desde las palabras mas comunes a las menos comunes)
    dicc_sorted = {}
    for k, v in sorted(dicc.items(), key=lambda item: item[1], reverse=True):
        dicc_sorted[k]=v

    return dicc_sorted


In [ ]:

#lista de palabras Stop_words (palabra que no agregan valor porque son conectores, preposiciones, etc...)

stop_words_dic = dic_stop_words(x_train)
stop_words = list(stop_words_dic.keys())[0:39]
print(stop_words)

In [ ]:
# Ahora crear un diccionario para comentarios positivos, neutral y negativos. No incluir palabras en "Stop_word"

# tweets --> texto a analizar (lista de los tweets limpios)
# cats --> lista que contiene solo las categorias (lista de las categorias limpias)
# categoria --> La unica categoria que quiero analizar
# stopwords --> parabras qu quiero que no sean tomadas en cuenta

def dic_palabras_comunes(tweets, cats, categoria, stopwords):

    dicc = {}
    for tweet, cat in zip(tweets, cats):
        if cat == categoria:
            for word in tweet.split(' '):
                if word not in stopwords:
                    if word in dicc:
                        dicc[word]+=1
                    else:
                        dicc[word]=1
    
    # Retorna un dic, pero ordenado de mayor a menor
    dicc_sorted={}
    for k, v in sorted(dicc.items(), key=lambda item: item[1], reverse=True):
        dicc_sorted[k]=v

    return dicc_sorted


In [ ]:
positivo_dicc = dic_palabras_comunes(x_train, y_train, 'positivo', stop_words)
neutral_dicc = dic_palabras_comunes(x_train, y_train, 'neutral', stop_words)
negativo_dicc = dic_palabras_comunes(x_train, y_train, 'negativo', stop_words)

#print(negativo_dicc)
#print(x_train)

In [ ]:
# Creando las listas de palabras unicas por lista

positivos = set(positivo_dicc.keys())
negativos = set(negativo_dicc.keys()) - positivos # neg - pos
neutrales = set(neutral_dicc.keys()) - positivos.union(negativos) # neu - pos - neg



#print(positivos.intersection(negativos).intersection(neutrales))
#print(len(positivos))
#print(len(negativos.intersection(positivos)))

In [ ]:

# Funcion FINAL clasificadora
def clasificador_sentimiento(tweet, stop_words, pos, neg, neu):  # pos, neg y neu: Son las listas de las palabras que ya estan separadas mas arriba
    
    #Eliminar stop_words
    words = tweet.split(' ')
    words = [word for word in words if word not in stop_words]

    if len(words) != 0:
        #Realiza conteo
        coincidencia_pos = set(pos).intersection(set(words))
        coincidencia_neg = set(neg).intersection(set(words))
        coincidencia_neu = set(neu).intersection(set(words))

        '''print(f'Positivo: {coincidencia_pos}')
        print(f'Negativos: {coincidencia_neg}')
        print(f'Neutral: {coincidencia_neu}')'''

        conteo_pos = len(coincidencia_pos)
        conteo_neg = len(coincidencia_neg)
        conteo_neu = len(coincidencia_neu)

        #calcular proporciones
        l_tweet = len(words)
        proporciones = [conteo_pos/l_tweet, conteo_neg/l_tweet, conteo_neu/l_tweet]
        maximo = max(proporciones)

        if proporciones[0] == maximo:
            cat = 'P'
        elif proporciones[1] == maximo:
            cat = 'N'
        elif proporciones[2] == maximo:
            cat = 'NEU'
            
    else:
        # Si todas las palabras son stop words
        cat = 'NEU'

    return cat


In [ ]:
idx = 9


# CLASIFICADOR
#clasificador_sentimiento(x_test[idx], stop_words, positivos, negativos, neutrales)

print(f'Tweet: {x_test[idx]} --> {y_test[idx]}\nCategoria: {clasificador_sentimiento(x_test[idx], stop_words, positivos, negativos, neutrales)}')


In [ ]:
# Probando la funcicon "clasificador_sentimiento" y su margen de acierto. (por ahora es muy baja)

def contador_aciertos(x_tweet, y_cat, stopWords, pos, neg, neu): # Return: [intentos, aciertos, desaciertos]

    intentos = 0
    aciertos = 0
    desaciertos = 0
    for i in range(len(x_test)):
        intentos += 1
        
        if clasificador_sentimiento(x_tweet[i], stopWords, pos, neg, neu)=='P' and y_cat[i] == 'positivo':
            aciertos += 1
        
        elif clasificador_sentimiento(x_tweet[i], stopWords, pos, neg, neu)=='N' and y_cat[i] == 'negativo':
            aciertos += 1
        
        elif clasificador_sentimiento(x_tweet[i], stopWords, pos, neg, neu)=='NEU' and y_cat[i] == 'neutral':
            aciertos += 1
        else: 
            desaciertos += 1

    return [intentos, aciertos, desaciertos] #[intentos, aciertos, desaciertos]


In [ ]:
list_acierto = contador_aciertos(x_test, y_test, stop_words, positivos, negativos, neutrales)

In [ ]:
print(list_acierto)
print(f'Acierto: {list_acierto[1]/list_acierto[0]:.2f}%')